In [1]:
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

import electricity
import util

In [2]:
utility = electricity.get_utility()
df = pd.json_normalize(utility)
df.columns

Index(['Utility.Number', 'Utility.Name', 'Utility.State', 'Utility.Type',
       'Demand.Summer Peak', 'Demand.Winter Peak', 'Sources.Generation',
       'Sources.Purchased', 'Sources.Other', 'Sources.Total', 'Uses.Retail',
       'Uses.Resale', 'Uses.No Charge', 'Uses.Consumed', 'Uses.Losses',
       'Uses.Total', 'Revenues.Retail', 'Revenue.Delivery', 'Revenue.Resale',
       'Revenue.Adjustments', 'Revenue.Transmission', 'Revenue.Other',
       'Revenue.Total', 'Retail.Residential.Revenue',
       'Retail.Residential.Sales', 'Retail.Residential.Customers',
       'Retail.Commercial.Revenue', 'Retail.Commercial.Sales',
       'Retail.Commercial.Customers', 'Retail.Industrial.Revenue',
       'Retail.Industrial.Sales', 'Retail.Industrial.Customers',
       'Retail.Transportation.Revenue', 'Retail.Transportation.Sales',
       'Retail.Transportation.Customers', 'Retail.Total.Revenue',
       'Retail.Total.Sales', 'Retail.Total.Customers'],
      dtype='str')

In [3]:
# df[["Uses.Retail", "Uses.Resale", "Uses.No Charge", "Uses.Consumed", "Uses.Losses", "Uses.Total"]]

In [4]:
ny_df = util.get_state_data('NC', df)
ny_df = util.prepare_data(ny_df)

ny_df.columns

Index(['Utility.Name', 'Utility.State', 'Utility.Type', 'Sources.Total',
       'Sources.Generation', 'Sources.Purchased', 'Sources.Other',
       'Retail.Residential.Revenue', 'Retail.Residential.Sales',
       'Retail.Residential.Customers', 'Retail.Industrial.Revenue',
       'Retail.Industrial.Sales', 'Uses.Retail', 'Uses.Losses', 'Uses.Total',
       'Demand.Summer Peak', 'Revenues.Retail', 'ResidentialUnitPrice',
       'IndustrialUnitPrice', 'IndustrialRevenueRatio', 'PriceSpread',
       'LoadFactor'],
      dtype='str')

In [5]:
# ny_df[ny_df.IndustrialPriceRatio.isna()]

In [6]:
def plot_scatterplot(df):
    fig = px.scatter(ny_df, x='LoadFactor', y='ResidentialUnitPrice', color="Utility.Type",
                    title="",
                    labels={
                            "LoadFactor": "System Efficiency (Load Factor)",
                            "ResidentialUnitPrice": "Residential Rate ($/MWh)"
                        },
                    hover_name="Utility.Name")
    fig.show()

plot_scatterplot(ny_df)

In [7]:
def plot_dumbbell(df):
    # Sort by spread to show the most "unfair" utilities at the top
    df_sorted = df[df.PriceSpread > 0].sort_values('PriceSpread', ascending=True).tail(10)

    fig = go.Figure()

    # 1. Add the lines connecting the dots
    for i, row in df_sorted.iterrows():
        fig.add_shape(
            type='line', x0=row['IndustrialUnitPrice'], x1=row['ResidentialUnitPrice'],
            y0=row['Utility.Name'], y1=row['Utility.Name'],
            line=dict(color='lightgrey', width=2)
        )

    # 2. Add Industrial dots
    fig.add_trace(go.Scatter(
        x=df_sorted['IndustrialUnitPrice'], y=df_sorted['Utility.Name'],
        mode='markers', name='Industrial Rate', marker=dict(color='#1f77b4', size=10)
    ))

    # 3. Add Residential dots
    fig.add_trace(go.Scatter(
        x=df_sorted['ResidentialUnitPrice'], y=df_sorted['Utility.Name'],
        mode='markers', name='Residential Rate', marker=dict(color='#d62728', size=10)
    ))

    fig.update_layout(title="Rate Disparity by Utility", xaxis_title="$/MWh", yaxis_title="")
    return fig

ps_df = ny_df[ny_df.IndustrialUnitPrice != 0]
ps_df = ps_df[ps_df.ResidentialUnitPrice != 0]

plot_dumbbell(ps_df)

In [8]:
def get_state_box_plot(df):
    # Compare Residential Price across Utility Types
    fig = px.box(df, x='Utility.Type', y='ResidentialUnitPrice', color="Utility.Type", 
                 points="all", hover_name="Utility.Name", hover_data=["Retail.Residential.Customers", "LoadFactor"],
                 title="Residential Price Distribution by Utility Ownership Type")
    #     plt.ylabel("Residential Rate ($/MWh)")
#     plt.xlabel("Ownership Model")
    fig.show()

get_state_box_plot(ny_df)

In [18]:
ny_df[ny_df["Utility.Type"] == "Retail Power Marketer"].LoadFactor.iloc[0]

np.float64(inf)

In [10]:
def plot_sankey(row):
    # 'row' is a single utility's data
    labels = ["Generation", "Purchased", "Other Source", "Total Sources", "Retail Sales", "Losses", "Other Uses"]
    
    # Convert raw values to percentages of the 'Total Sources'
    total = row['Sources.Total']
    sources_gen_pct = (row['Sources.Generation'] / total) * 100
    sources_pur_pct = (row['Sources.Purchased'] / total) * 100
    sources_oth_pct = (row['Sources.Other'] / total) * 100
    uses_retail_pct = (row['Uses.Retail'] / total) * 100
    uses_loss_pct = (row['Uses.Losses'] / total) * 100

    fig = go.Figure(data=[go.Sankey(
        node = dict(
          pad = 15, thickness = 20,
          line = dict(color = "black", width = 0.5),
          label = labels
        ),
        link = dict(
          source = [0, 1, 2, 3, 3, 3], 
          target = [3, 3, 3, 4, 5, 6], 
          value = [
              sources_gen_pct, 
              sources_pur_pct, 
              sources_oth_pct,
              uses_retail_pct, 
              uses_loss_pct,
              max(0, 100 - (uses_retail_pct + uses_loss_pct))
          ]
      ))])

    fig.update_layout(title_text=f"Energy Flow %: {row['Utility.Name']}")
    return fig

plot_sankey(ny_df.iloc[99])

In [11]:
ny_df.iloc[67]

Utility.Name                    Town of Pinetops - (NC)
Utility.State                                        NC
Utility.Type                                  Municipal
Sources.Total                                   20223.0
Sources.Generation                                  0.0
Sources.Purchased                               20223.0
Sources.Other                                       0.0
Retail.Residential.Revenue                       1053.0
Retail.Residential.Sales                         7004.0
Retail.Residential.Customers                      622.0
Retail.Industrial.Revenue                        1078.0
Retail.Industrial.Sales                          8959.0
Uses.Retail                                     19405.0
Uses.Losses                                       818.0
Uses.Total                                      20223.0
Demand.Summer Peak                                  3.3
Revenues.Retail                                  2660.0
ResidentialUnitPrice                         150

In [12]:
# import plotly.graph_objects as go
# fig = go.Figure(go.Scatter(x=[0, 1], y=[10, 6], mode='lines+markers+text', 
#                            text=['start', 'end'], textposition=['middle left', 'middle right']))
# fig.add_shape(type='line', x0=0, x1=0, y0=0, y1=1, xref='x', yref='paper')
# fig.add_shape(type='line', x0=1, x1=1, y0=0, y1=1, xref='x', yref='paper')
# fig.show()